In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import torch

# Detect GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_AVAILABLE = device.type == 'cuda'
print(f"Using device: {device}")
if GPU_AVAILABLE:
    print(f"GPU: {torch.cuda.get_device_name(0)}")


class TorchKNN:
    """GPU-accelerated KNN classifier using PyTorch."""

    def __init__(self, n_neighbors=5, device='cpu', batch_size=1024):
        self.k = n_neighbors
        self.device = device
        self.batch_size = batch_size

    def fit(self, X, y):
        self.X_train = torch.tensor(np.asarray(X), dtype=torch.float32).to(self.device)
        self.y_train = torch.tensor(np.asarray(y), dtype=torch.long).to(self.device)
        return self

    def predict(self, X):
        X_t = torch.tensor(np.asarray(X), dtype=torch.float32).to(self.device)
        preds = []
        for i in range(0, len(X_t), self.batch_size):
            batch = X_t[i : i + self.batch_size]
            dists = torch.cdist(batch, self.X_train)          # (batch, n_train)
            knn_idx = dists.topk(self.k, largest=False).indices  # (batch, k)
            knn_labels = self.y_train[knn_idx]                  # (batch, k)
            pred = knn_labels.mode(dim=1).values                # majority vote
            preds.append(pred)
        return torch.cat(preds).cpu().numpy()

    def score(self, X, y):
        preds = self.predict(X)
        return float((preds == np.asarray(y)).mean())


Using device: cuda
GPU: NVIDIA GeForce RTX 3060


<h1>Data Cleaning</h1>

In [2]:
drop_cols = [
    'device_name', 'device_mac', 'label_full',
    'label1', 'label2', 'label3', 'label4',
    'timestamp', 'timestamp_start', 'timestamp_end',
    'log_data-types', 'log_interval-messages',
    'network_ips_all', 'network_ips_dst', 'network_ips_src',
    'network_macs_all', 'network_macs_dst', 'network_macs_src',
    'network_ports_all', 'network_ports_dst', 'network_ports_src',
    'network_protocols_all', 'network_protocols_dst', 'network_protocols_src',
]


# load data and set up x_train, y_train, x_test, y_test
benignData = pd.read_csv("benignDataCombinedFile.csv")
benignData["label"] = "benign"

attackData = []

for i in range(1, 10):
    dataFile_a = pd.read_csv(f'dataSamples/attack_data/attack_samples_{i}sec.csv')
    dataFile_a = dataFile_a.drop(columns=drop_cols, errors='ignore')
    dataFile_a["label"] = "attack"
    attackData.append(dataFile_a)

combinedData = pd.concat([benignData] + attackData, ignore_index=True)

<h1>KNN modeling</h1>

<h2>Training</h2>

In [3]:
X = combinedData.drop(columns=['label'] + drop_cols, errors='ignore')
y = combinedData['label'].astype(str).str.lower().map({'benign': 0, 'attack': 1})

# Keep only numeric features
X = X.select_dtypes(include=[np.number]).fillna(0)

# Remove rows with unexpected/missing labels
valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

knn_classifier = TorchKNN(n_neighbors=5, device=str(device))
knn_classifier.fit(X_train.values, y_train.values)

device_used = f"{'GPU' if GPU_AVAILABLE else 'CPU'} (PyTorch — {device})"
print(f"Model trained on {device_used}")


Model trained on GPU (PyTorch — cuda)


<h2>Evaluation</h2>

In [4]:
y_pred = knn_classifier.predict(X_test.values)
y_true = y_test.values

accuracy = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred)

print(f"Device: {device_used}")
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)


Device: GPU (PyTorch — cuda)
Accuracy: 0.9914
Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     80135
           1       0.99      0.98      0.99     53730

    accuracy                           0.99    133865
   macro avg       0.99      0.99      0.99    133865
weighted avg       0.99      0.99      0.99    133865



Logistic Regression

In [5]:
train_accuracy = knn_classifier.score(X_train.values, y_train.values)
test_accuracy  = knn_classifier.score(X_test.values,  y_test.values)
gap = train_accuracy - test_accuracy

print(f"Device:         {device_used}")
print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")
print(f"Gap:            {gap:.4f}")


Device:         GPU (PyTorch — cuda)
Train accuracy: 0.9937
Test accuracy:  0.9914
Gap:            0.0023
